# 第34章 折线图（plot）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 3 / 12 步：表达趋势与类别比较**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 绘图结构（Figure / Axes）  →  **本章任务：** 折线图（plot）  →  **下一步：** 柱状图（bar / barh）
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

当我们手里只有一列按时间或顺序排列的数字时，散点图太零散、柱状图又偏向强调单个体量，而折线图最擅长把“从前往后”的连续变化一目了然地展现出来。


## 本章目标

学完本章，你将能够：

- **理解**：理解「折线图（plot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「折线图（plot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「折线图（plot）」并读出其中的结论。


## 适用场景

**背景引入**：当我们手里只有一列按时间或顺序排列的数字时，散点图太零散、柱状图又偏向强调单个体量，而折线图最擅长把“从前往后”的连续变化一目了然地展现出来。比如月底做销售复盘，领导真正关心的往往不是某一个月卖了多少，而是整个上半年是在上升还是回落——折线图的走向一眼就能回答这个问题。学会折线图，你才能把“趋势”这个抽象分析需求变成一张与人沟通无障碍的图。 打个比方：折线图就像把每个月的销量串成一条上山的路——路是往上走还是往下走，一眼就能看出趋势；要只盯某一个月，就像只盯着路上的一块石头，看不清整条路的方向。

X轴具有自然顺序，重点是观察连续变化、增长速度或周期。


## 数据结构

一列有序时间或阶段，一列或多列同单位指标；缺失时间点需要显式处理。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 marker='o' 改为 marker='s'，观察标记形状变化
2. 调整 linewidth 参数（如 0.5 或 3.5），说明线条粗细对可读性的影响
3. 修改 linestyle 为 '--'，对比虚线与实线的视觉效果


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`ax.plot()`、`ax.set()`、`ax.grid()` | X轴具有自然顺序，重点是观察连续变化、增长速度或周期。 | 用折线连接无顺序类别 |
| 进阶变体 | `plt.subplots()`、`ax.plot()`、`ax.axhline()`、`ax.set()` | 在基础图表上增加分组、注释、布局或交互 | 多条线颜色过近 |
| 关键参数 | `marker` | 观测点 | 用折线连接无顺序类别 |
| 关键参数 | `linestyle` | 线型 | 多条线颜色过近 |
| 关键参数 | `linewidth` | 线宽 | 时间缺失却直接连接 |
| 关键参数 | `label` | 序列名称 | 用折线连接无顺序类别 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-34 -->
### 数学推导｜趋势图中的变化量与增长率

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先算绝对变化。** $\Delta x_t=x_t-x_{t-1}$ 保留原单位。

**第 2 步｜再除以前一期形成相对变化。** 先写倍率 $r_t=x_t/x_{t-1}$，增长率就是 $g_t=r_t-1$。

**第 3 步｜多期增长要连乘。** 从 0 期到 $T$ 期的累计增长满足

$$
\frac{x_T}{x_0}=\prod_{t=1}^{T}(1+g_t)
$$

因此不能把多期百分比简单相加，除非变化都很小且只做近似。

**把上面的关系收束为本章计算式：**

$$
\Delta x_t=x_t-x_{t-1},\qquad g_t=\frac{x_t-x_{t-1}}{x_{t-1}}
$$

**符号解释：** $\Delta x_t$ 是绝对变化，$g_t$ 是环比增长率。

**代码对应：** 排序后使用 `diff()` 与 `pct_change()`，再把结果放进 Hover 或注释。

**使用边界：** 当上一期为 0 或时间间隔不一致时，增长率需要特殊处理。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = (
    completed["InvoiceDate"].dt.to_period("M").astype("string")
)

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
# 简化假设：利润在 15% 基础上轻微递增，让折线图呈现波动趋势
profit = sales * (0.15 + 0.02 * np.linspace(0, 1, len(sales)))

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(
        rows["Country"].astype(str),
        rows["flow"],
        values=rows["amount"].abs(),
        aggfunc="sum",
    )
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales, marker="o", linewidth=2.2, color="#1a73e8")
ax.set(title="上半年销售额趋势", xlabel="月份", ylabel="销售额（万元）")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：把 29.4 的折线图从“销售额”换成“订单数”，其余编码保持不变，观察数据字段变化对图形的影响。

**提示**：基础图表用的是 ax.plot(months, sales, ...)；这里把第二个参数 sales 改成 orders，并在标题、纵轴标签里同步改成“订单数”。运行后对比两条折线的数值量级与起伏节奏有何不同。


In [ ]:
try:
    pass
    # 请在下方填写代码
    # 目标：绘制 months 对 orders 的折线图，观察与 sales 的图形差异。
    # ______(1)______ 在下方用 ax.plot 绘制 months 与 orders，marker='o'，linewidth=2.2
    # ______(2)______ ax.set 标题“上半年订单数趋势”、xlabel“月份”、ylabel“订单数（单）”

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

sales_index = sales / sales[0] * 100
profit_index = profit / profit[0] * 100
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales_index, marker="o", label="销售额指数")
ax.plot(months, profit_index, marker="s", label="利润指数")
ax.axhline(100, color="#9aa0a6", linestyle="--", linewidth=1)
ax.set(title="利润增长快于销售额", ylabel="指数（1月=100）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 参数说明

- marker：观测点
- linestyle：线型
- linewidth：线宽
- label：序列名称


## 结果解读

先读总体方向，再找峰谷、转折和序列间差距；不能把连接线误解为未观测区间的真实数据。


## 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月", "4月"]
_demo_sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月"]
_demo_sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 用折线连接无顺序类别
- 多条线颜色过近
- 时间缺失却直接连接


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：把「销售额」趋势换成「订单量」趋势，比较两者走势
    # 【目标】换一个指标，练习观察「两张图走势有何不同」。
    #   销售额看的是「钱」，订单量看的是「笔数」——两者可能同步，也可能背离。
    import matplotlib.pyplot as plt

    # 起点示例（已可运行）：y 换成 orders，标记换成星号以区分。
    #   同时保留顶部/右侧无边框、横向网格的风格，让图更清爽。
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(months, orders, marker="*", linewidth=2.2, color="#e8710a")
    ax.set(title="上半年订单量趋势", xlabel="月份", ylabel="订单量（笔）")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    plt.show()

    # ---- 反思记录：换指标后，用三句话写下观察 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用折线位置和斜率表达有序时间上的趋势、转折和多序列差异。


### 你已经掌握

- 判断折线图（plot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `marker` | 观测点 |
| `linestyle` | 线型 |
| `linewidth` | 线宽 |
| `label` | 序列名称 |


### 需要注意

- 用折线连接无顺序类别
- 多条线颜色过近
- 时间缺失却直接连接


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# ===== 练一练参考答案 =====
# 数据字段从 sales 换成 orders，其余编码保持不变即可。
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, orders, marker="o", linewidth=2.2, color="#1a73e8")
ax.set(title="上半年订单数趋势", xlabel="月份", ylabel="订单数（单）")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

# 恢复与 orders 长度一致的月份序列（前面的示例可能已改写成 3 个月）。
months = monthly_summary.index.to_numpy()

growth = np.diff(orders)
fastest = int(growth.argmax()) + 1
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, orders, marker="o", color="#188038")
ax.scatter(months[fastest], orders[fastest], s=90, color="#d93025", zorder=3)
ax.annotate(
    f"增加 {growth[fastest - 1]} 单",
    (months[fastest], orders[fastest]),
    xytext=(-45, 25),
    textcoords="offset points",
    arrowprops={"arrowstyle": "->"},
)
ax.set(title="订单量趋势及最大增量", ylabel="订单数")
fig.tight_layout()
plt.show()
